In [2]:
import pandas as pd
import joblib, os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv('../data/raw/diabetes_012_health_indicators_BRFSS2015.csv')  # or chronic_sample.csv

features = ['HighBP', 'HighChol', 'BMI', 'Smoker', 'PhysActivity', 'GenHlth', 'Age', 'Sex']
target = 'Diabetes_012'

model_df = df[features + [target]].copy()
print(model_df.shape)
print("Missing:", model_df.isnull().sum().sum())
print("Duplicates:", model_df.duplicated().sum(), "(kept, not removed)")
print((model_df[target].value_counts(normalize=True).sort_index() * 100).round(2))

(253680, 9)
Missing: 0
Duplicates: 197586 (kept, not removed)
Diabetes_012
0.0    84.24
1.0     1.83
2.0    13.93
Name: proportion, dtype: float64


In [3]:
X = model_df.drop(columns=[target])
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train %:\n", (y_train.value_counts(normalize=True).sort_index() * 100).round(2))
print("Test %:\n", (y_test.value_counts(normalize=True).sort_index() * 100).round(2))

Train: (202944, 8) Test: (50736, 8)
Train %:
 Diabetes_012
0.0    84.24
1.0     1.83
2.0    13.93
Name: proportion, dtype: float64
Test %:
 Diabetes_012
0.0    84.24
1.0     1.83
2.0    13.93
Name: proportion, dtype: float64


In [4]:
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Test Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Test Accuracy: 83.01 %
              precision    recall  f1-score   support

         0.0       0.87      0.95      0.91     42741
         1.0       0.04      0.01      0.01       926
         2.0       0.41      0.22      0.29      7069

    accuracy                           0.83     50736
   macro avg       0.44      0.39      0.40     50736
weighted avg       0.79      0.83      0.80     50736

[[40549    87  2105]
 [  797     5   124]
 [ 5461    48  1560]]


In [5]:
model = RandomForestClassifier(n_estimators=100, class_weight='balanced',
                               random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Test Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Test Accuracy: 61.75 %
              precision    recall  f1-score   support

         0.0       0.92      0.64      0.76     42741
         1.0       0.02      0.21      0.04       926
         2.0       0.28      0.52      0.37      7069

    accuracy                           0.62     50736
   macro avg       0.41      0.46      0.39     50736
weighted avg       0.82      0.62      0.69     50736

[[27454  6359  8928]
 [  336   198   392]
 [ 1974  1415  3680]]


In [6]:
y_binary = y.replace({0: 0, 1: 1, 2: 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.2, stratify=y_binary, random_state=42
)

model = RandomForestClassifier(n_estimators=100, class_weight='balanced',
                               random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Test Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Test Accuracy: 71.57 %
              precision    recall  f1-score   support

         0.0       0.92      0.73      0.81     42741
         1.0       0.31      0.65      0.42      7995

    accuracy                           0.72     50736
   macro avg       0.61      0.69      0.62     50736
weighted avg       0.82      0.72      0.75     50736

[[31086 11655]
 [ 2771  5224]]


In [7]:
os.makedirs('../models', exist_ok=True)
joblib.dump(model, '../models/chronic_model.pkl')
joblib.dump(list(X_train.columns), '../models/chronic_features.pkl')
print("Saved. Feature order:", list(X_train.columns))

Saved. Feature order: ['HighBP', 'HighChol', 'BMI', 'Smoker', 'PhysActivity', 'GenHlth', 'Age', 'Sex']


# Additional Experiment: Random Forest Using All 21 Features

In this experiment, Random Forest is trained using all 21 input features
from the chronic health dataset.

The target remains Diabetes_012 with three classes:
0 - No Diabetes
1 - Prediabetes
2 - Diabetes

An 80:20 stratified train-test split with random_state=42 is used.
This experiment allows comparison with the Decision Tree model
trained using the same features and dataset.

In [8]:
# Load dataset for the all-feature Random Forest experiment

rf_all_df = pd.read_csv("../data/processed/chronic_sample.csv")

print("Dataset shape:", rf_all_df.shape)
print("Number of features:", rf_all_df.shape[1] - 1)

Dataset shape: (20000, 22)
Number of features: 21


In [9]:
# Separate features and target

rf_all_X = rf_all_df.drop(columns=["Diabetes_012"])
rf_all_y = rf_all_df["Diabetes_012"]

print("Feature shape:", rf_all_X.shape)
print("Target shape:", rf_all_y.shape)

Feature shape: (20000, 21)
Target shape: (20000,)


In [10]:
# Stratified train-test split

from sklearn.model_selection import train_test_split

rf_all_X_train, rf_all_X_test, rf_all_y_train, rf_all_y_test = train_test_split(
    rf_all_X,
    rf_all_y,
    test_size=0.2,
    random_state=42,
    stratify=rf_all_y
)

print("Training samples:", rf_all_X_train.shape[0])
print("Testing samples:", rf_all_X_test.shape[0])

Training samples: 16000
Testing samples: 4000


In [11]:
# Train Random Forest using all 21 features

from sklearn.ensemble import RandomForestClassifier

rf_all_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_all_model.fit(rf_all_X_train, rf_all_y_train)

print("Random Forest trained successfully!")

Random Forest trained successfully!


In [12]:
# Make predictions

rf_all_pred = rf_all_model.predict(rf_all_X_test)

# Calculate accuracy

rf_all_accuracy = accuracy_score(rf_all_y_test, rf_all_pred)

print("Random Forest Accuracy:",
      round(rf_all_accuracy * 100, 2), "%")

# Classification report

print("\nRandom Forest Classification Report:\n")

print(classification_report(
    rf_all_y_test,
    rf_all_pred,
    target_names=[
        "No Diabetes",
        "Prediabetes",
        "Diabetes"
    ],
    zero_division=0
))

Random Forest Accuracy: 84.45 %

Random Forest Classification Report:

              precision    recall  f1-score   support

 No Diabetes       0.86      0.97      0.91      3375
 Prediabetes       0.00      0.00      0.00        75
    Diabetes       0.49      0.19      0.27       550

    accuracy                           0.84      4000
   macro avg       0.45      0.39      0.39      4000
weighted avg       0.80      0.84      0.81      4000



In [13]:
# Class-weighted Random Forest using all 21 features

rf_balanced_model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_balanced_model.fit(rf_all_X_train, rf_all_y_train)

rf_balanced_pred = rf_balanced_model.predict(rf_all_X_test)

print("Balanced Random Forest Accuracy:",
      round(accuracy_score(rf_all_y_test, rf_balanced_pred) * 100, 2), "%")

print("\nClassification Report:\n")

print(classification_report(
    rf_all_y_test,
    rf_balanced_pred,
    target_names=[
        "No Diabetes",
        "Prediabetes",
        "Diabetes"
    ],
    zero_division=0
))

Balanced Random Forest Accuracy: 80.4 %

Classification Report:

              precision    recall  f1-score   support

 No Diabetes       0.91      0.87      0.89      3375
 Prediabetes       0.00      0.00      0.00        75
    Diabetes       0.38      0.53      0.44       550

    accuracy                           0.80      4000
   macro avg       0.43      0.47      0.44      4000
weighted avg       0.82      0.80      0.81      4000



In [15]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("Evaluation metrics imported successfully!")

Evaluation metrics imported successfully!


In [16]:
# Compare Random Forest experiments

rf_comparison = pd.DataFrame({
    "Model": [
        "Default Random Forest",
        "Balanced Random Forest"
    ],
    "Accuracy": [
        accuracy_score(rf_all_y_test, rf_all_pred),
        accuracy_score(rf_all_y_test, rf_balanced_pred)
    ],
    "Macro Precision": [
        precision_score(rf_all_y_test, rf_all_pred, average="macro", zero_division=0),
        precision_score(rf_all_y_test, rf_balanced_pred, average="macro", zero_division=0)
    ],
    "Macro Recall": [
        recall_score(rf_all_y_test, rf_all_pred, average="macro", zero_division=0),
        recall_score(rf_all_y_test, rf_balanced_pred, average="macro", zero_division=0)
    ],
    "Macro F1": [
        f1_score(rf_all_y_test, rf_all_pred, average="macro", zero_division=0),
        f1_score(rf_all_y_test, rf_balanced_pred, average="macro", zero_division=0)
    ]
})

rf_comparison.round(4)

,Model,Accuracy,Macro Precision,Macro Recall,Macro F1
0,Default Random Forest,0.8445,0.4522,0.3854,0.3946
1,Balanced Random Forest,0.8040,0.4289,0.4653,0.4428


In [17]:
# Save Random Forest comparison results

rf_results = {
    "Default Random Forest": {
        "Accuracy": accuracy_score(rf_all_y_test, rf_all_pred),
        "Macro F1": f1_score(rf_all_y_test, rf_all_pred, average="macro"),
        "Macro Recall": recall_score(rf_all_y_test, rf_all_pred, average="macro")
    },
    "Balanced Random Forest": {
        "Accuracy": accuracy_score(rf_all_y_test, rf_balanced_pred),
        "Macro F1": f1_score(rf_all_y_test, rf_balanced_pred, average="macro"),
        "Macro Recall": recall_score(rf_all_y_test, rf_balanced_pred, average="macro")
    }
}

print(rf_results)

{'Default Random Forest': {'Accuracy': 0.8445, 'Macro F1': 0.39455828204236393, 'Macro Recall': 0.38537373737373737}, 'Balanced Random Forest': {'Accuracy': 0.804, 'Macro F1': 0.4428037979570085, 'Macro Recall': 0.46525252525252525}}
